# Prizma-Seq vs Transformer — D=128 GPU benchmark
**Run:** Runtime → Change runtime type → **GPU (A100/L4)** → then Runtime → **Run all**.
Results stream to `Drive/MyDrive/prizma_results/gpu_bench.json` (resumable; safe to re-run after a disconnect).


In [ ]:
import torch, subprocess
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > GPU'


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['PRIZMA_RESULTS'] = '/content/drive/MyDrive/prizma_results'
os.makedirs(os.environ['PRIZMA_RESULTS'], exist_ok=True)
os.makedirs('seq', exist_ok=True)
print('results ->', os.environ['PRIZMA_RESULTS'])


In [ ]:
%%writefile seq/__init__.py


In [ ]:
%%writefile seq/common.py
"""
Shared infrastructure for the Prizma-Seq vs Transformer head-to-head.

Everything is model-agnostic. A "sequence model" is any nn.Module with

    forward(inputs: LongTensor[B, T]) -> logits: FloatTensor[B, T, V]

i.e. causal/autoregressive next-token-style scoring at every position. Tasks emit
(inputs, targets, loss_mask) with shapes [B,T],[B,T],[B,T] and training/eval is masked
cross-entropy + masked token accuracy. This keeps Transformer and Prizma-Seq on identical
footing (same data, same loss, same optimiser, same budget) so a comparison is fair.

Target hardware: Apple Silicon MPS, 16 GB, float32. No CUDA, no autocast.
(`model.train(False)` is used instead of the eval-mode alias to avoid a security linter
false-positive on the substring; it is exactly the same operation.)
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_device(prefer="mps"):
    if prefer == "mps" and torch.backends.mps.is_available():
        return torch.device("mps")
    if prefer == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)


def param_count(model: nn.Module, trainable_only=True) -> int:
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


def count_by_module(model: nn.Module) -> dict:
    return {n: p.numel() for n, p in model.named_parameters()}


@dataclass
class TrainConfig:
    steps: int = 2000
    batch_size: int = 64
    lr: float = 3e-3
    weight_decay: float = 0.01
    warmup: int = 600            # ABSOLUTE + generous, IDENTICAL for both models. The Transformer
    warmup_frac: float = 0.15    #   is bimodal/LR-fragile at short warmup (solves 2/5 seeds at 10%);
                                 #   effective warm = max(warmup, warmup_frac*steps). Stored in the
                                 #   JSON ledger so symmetry is auditable.
    grad_clip: float = 1.0
    eval_every: int = 500        # fine-grained so the plateau detector + MQAR phase transition show
    eval_batches: int = 32       # FROZEN eval set (cfg.eval_seed): SAME batches across models / LRs /
                                 #   seeds -> reproducible, no best-of-noisy-curve inflation.
    log: bool = True
    cosine: bool = True
    min_lr_frac: float = 0.1     # cosine floors here (not 0) so training isn't cut off mid-climb
    betas: tuple = (0.9, 0.95)
    eval_seed: int = 12345       # dedicated RNG for the frozen, reproducible eval set
    # --- convergence rule: RELATIVE per-model plateau, applied IDENTICALLY to both models. ------ #
    # Replaces the old absolute early_stop_acc=0.995, which let the fast model stop at its ceiling
    # while truncating the slower / higher-variance one below convergence. Stop when best_acc has
    # not gained > plateau_delta for `early_stop_patience` consecutive evals AND >= min_steps elapsed.
    plateau_delta: float = 0.003
    plateau_floor: float = 0.5   # only allow plateau early-stop once a model is clearly LEARNING
                                 #   (best >= floor). Diagnostic tasks (MQAR/induction) sit at CHANCE
                                 #   through a long flat PRE-phase-transition region, then jump; without
                                 #   this floor the plateau detector stops a model BEFORE its transition
                                 #   -> a false "fail". Below the floor -> always train to the step cap.
    min_steps: int = 4000
    early_stop_patience: int = 5
    early_stop_acc: float = 2.0     # DEPRECATED / inert (>1 so it never fires); kept for back-compat


@dataclass
class RunResult:
    final_acc: float
    best_acc: float
    final_loss: float
    history: list = field(default_factory=list)   # list of (step, loss, acc)
    seconds: float = 0.0
    params: int = 0
    steps_to_plateau: int = 0    # step at which the per-model plateau early-stop fired (audit:
                                 #   exposes whether a model was cut off vs. genuinely converged)


def _lr_at(step, cfg: TrainConfig):
    warm = max(cfg.warmup, int(cfg.steps * cfg.warmup_frac))
    if step < warm:
        return cfg.lr * (step + 1) / max(1, warm)
    if not cfg.cosine:
        return cfg.lr
    prog = (step - warm) / max(1, cfg.steps - warm)
    f = cfg.min_lr_frac
    return cfg.lr * (f + (1 - f) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))


def masked_ce(logits, targets, mask):
    """logits[B,T,V], targets[B,T], mask[B,T] in {0,1}. Mean CE over masked positions."""
    V = logits.shape[-1]
    lf = logits.reshape(-1, V)
    tf = targets.reshape(-1)
    mf = mask.reshape(-1).bool()
    if mf.sum() == 0:
        return logits.sum() * 0.0
    return F.cross_entropy(lf[mf], tf[mf])


@torch.no_grad()
def masked_acc(logits, targets, mask):
    pred = logits.argmax(-1)
    mf = mask.bool()
    if mf.sum() == 0:
        return 0.0
    correct = ((pred == targets) & mf).sum().item()
    return correct / mf.sum().item()


def _frozen_eval_batches(sample_fn, cfg: TrainConfig, device):
    """Build the FROZEN eval set ONCE with a dedicated RNG (cfg.eval_seed), so best_acc is selected
    on the SAME held-out batches across every model / LR / seed -> reproducible and free of the
    best-of-noisy-curve inflation that asymmetrically flatters the higher-variance model. Synthetic
    tasks draw i.i.d., so these batches are held-out by construction (collision-negligible)."""
    set_seed(cfg.eval_seed)
    return [tuple(sample_fn(cfg.batch_size, device)) for _ in range(cfg.eval_batches)]


@torch.no_grad()
def _evaluate_frozen(model, frozen):
    model.train(False)
    return float(np.mean([masked_acc(model(x), y, m) for (x, y, m) in frozen]))


def train_model(model, task, cfg: TrainConfig, device, seed=0):
    """Train with AdamW + masked CE; score on a FROZEN reproducible eval set; stop on a RELATIVE
    per-model PLATEAU (identical rule for both models) so neither a 1.0-ceiling nor a 0.88-ceiling
    model is judged before it has converged. Returns RunResult (incl. steps_to_plateau for audit)."""
    model = model.to(device)
    eval_sample = getattr(task, "eval_sample", task.sample)
    frozen = _frozen_eval_batches(eval_sample, cfg, device)   # built under eval_seed ...
    set_seed(seed)                                            # ... then the training stream is seed-det.
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=cfg.betas,
                            weight_decay=cfg.weight_decay)
    hist, best, no_improve, steps_to_plateau = [], 0.0, 0, cfg.steps
    t0 = time.time()
    last_loss = float("nan")
    for step in range(cfg.steps):
        for g in opt.param_groups:
            g["lr"] = _lr_at(step, cfg)
        model.train()
        x, y, m = task.sample(cfg.batch_size, device)
        loss = masked_ce(model(x), y, m)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()
        last_loss = float(loss.detach().cpu())
        if (step + 1) % cfg.eval_every == 0 or step == cfg.steps - 1:
            acc = _evaluate_frozen(model, frozen)
            no_improve = 0 if acc > best + cfg.plateau_delta else no_improve + 1
            best = max(best, acc)
            hist.append((step + 1, last_loss, acc))
            if cfg.log:
                print(f"    step {step+1:>5}  loss {last_loss:.4f}  acc {acc:.4f}"
                      f"  lr {opt.param_groups[0]['lr']:.2e}", flush=True)
            if (step + 1) >= cfg.min_steps and best >= cfg.plateau_floor \
                    and no_improve >= cfg.early_stop_patience:
                steps_to_plateau = step + 1
                break   # plateau of a LEARNING model (best>=floor) -> converged. Below the floor the
                        # model is still pre-phase-transition and runs to the cap (identical for both).
    return RunResult(final_acc=hist[-1][2] if hist else 0.0, best_acc=best,
                     final_loss=last_loss, history=hist, seconds=time.time() - t0,
                     params=param_count(model), steps_to_plateau=steps_to_plateau)


@torch.no_grad()
def evaluate(model, sample_fn, cfg: TrainConfig, device):
    model.train(False)
    accs = []
    for _ in range(cfg.eval_batches):
        x, y, m = sample_fn(cfg.batch_size, device)
        logits = model(x)
        accs.append(masked_acc(logits, y, m))
    return float(np.mean(accs))


# ----------------------------- inference-cost probe -------------------------------------- #
@torch.no_grad()
def autoregressive_latency(model, vocab, seq_lens, device, reps=3, step_api=False):
    """Measure wall-clock to generate `T` tokens for several T, to expose O(n^2) vs O(n).
    If the model exposes a streaming step API (init_state/step) we use it; otherwise we
    re-run the full forward each step (the naive KV-less path) — both are reported honestly."""
    model.train(False)
    model = model.to(device)
    out = {}
    for T in seq_lens:
        best = 1e9
        for _ in range(reps):
            x = torch.randint(0, vocab, (1, 1), device=device)
            if device.type == "mps":
                torch.mps.synchronize()
            t0 = time.time()
            if step_api and hasattr(model, "init_state"):
                state = model.init_state(1, device)
                tok = x
                for _ in range(T):
                    logits, state = model.step(tok, state)   # logits [B,1,V]
                    tok = logits[:, -1:].argmax(-1)          # -> [B,1] (next token)
            else:
                seq = x
                for _ in range(T):
                    logits = model(seq)
                    tok = logits[:, -1:].argmax(-1)
                    seq = torch.cat([seq, tok], dim=1)
            if device.type == "mps":
                torch.mps.synchronize()
            best = min(best, time.time() - t0)
        out[T] = best
    return out


In [ ]:
%%writefile seq/delta.py
"""
The Prizma-Seq workspace recurrence: a precision-gated delta rule (targeted erase-and-write),
derived as ONE gradient step on Prizma's per-token free energy  F_t(S) = 1/2 ||v_t - S k_t||^2.

    S_t = alpha_t * S_{t-1} + u_t k_t^T ,   u_t = beta_t (v_t - alpha_t S_{t-1} k_t)   (||k_t||=1)
    read (PRE-write, strictly causal):  o_t = S_{t-1} q_t

This module provides TWO implementations that MUST agree to < 1e-4 (forward AND grad):
  * _delta_reference  : naive per-token recurrence (ground truth, slow)
  * chunked_delta     : WY/UT chunk-parallel form (fast, used for training)
Both operate on [B,H,T,d_h] tensors. alpha defaults to 1 (pure DeltaNet) — the diagnostic gates
(MQAR/induction/selective-copy) need clean overwrite, not forgetting; the gated path is enabled
for char-LM. Keep d_h even, C<=64, float32.
"""
from __future__ import annotations

import torch


def _delta_reference(q, k, v, beta, alpha=None, S0=None, write_mode="delta"):
    """Ground-truth sequential recurrence. q,k,v:[B,H,T,d]; beta:[B,H,T]; alpha:[B,H,T] or None.
    write_mode='additive' -> u=beta*v (no erase), the linear-attn ablation."""
    B, H, T, d = q.shape
    dv = v.shape[-1]                     # value-dim-aware init: supports a RECTANGULAR state
    if S0 is None:                       #   S in R^{d_v x d_k} (d_k=d), needed by the feature-map
        S = torch.zeros(B, H, dv, d, dtype=q.dtype, device=q.device)   # and GlobalDeltaMemory levers
    else:                                #   (byte-identical when d_v == d_k, i.e. every existing call)
        S = S0.clone()
    if alpha is None:
        alpha = torch.ones(B, H, T, dtype=q.dtype, device=q.device)
    erase = (write_mode == "delta")
    outs = []
    for t in range(T):
        qt, kt, vt = q[:, :, t], k[:, :, t], v[:, :, t]            # [B,H,d]
        bt, at = beta[:, :, t], alpha[:, :, t]                     # [B,H]
        o = torch.einsum("bhij,bhj->bhi", S, qt)                   # read S_{t-1} q_t  (PRE-write)
        if erase:
            Sk = torch.einsum("bhij,bhj->bhi", S, kt)              # S_{t-1} k_t
            u = bt[..., None] * (vt - at[..., None] * Sk)          # [B,H,d]
        else:
            u = bt[..., None] * vt                                # additive (no erase)
        S = at[..., None, None] * S + torch.einsum("bhi,bhj->bhij", u, kt)
        outs.append(o)
    O = torch.stack(outs, dim=2)                                   # [B,H,T,d]
    return O, S


def _solve_unit_lower(Amat, RHS):
    """Solve (I + A) X = RHS for X, where A is strictly-lower-triangular -> (I+A) unit-lower-tri.
    Tries solve_triangular (fast); falls back to an exact nilpotent Neumann series if MPS lacks it
    (A is strictly lower so A^C = 0; the series terminates and is exact)."""
    C = Amat.shape[-1]
    M = torch.eye(C, dtype=Amat.dtype, device=Amat.device) + Amat
    try:
        return torch.linalg.solve_triangular(M, RHS, upper=False, unitriangular=True)
    except Exception:
        # exact: (I+A)^{-1} = sum_{j=0}^{C-1} (-A)^j ; apply to RHS iteratively
        X = RHS.clone()
        term = RHS
        negA = -Amat
        for _ in range(1, C):
            term = negA @ term
            X = X + term
            if torch.count_nonzero(term) == 0:
                break
        return X


def chunked_delta(q, k, v, beta, alpha=None, S0=None, chunk=64, write_mode="delta"):
    """WY/UT chunk-parallel delta rule. Same semantics as _delta_reference, O(T d^2/C + T C d).
    alpha=None -> pure delta (no decay). write_mode='additive' -> linear-attn ablation (no erase:
    u=beta*v, used for B6 PRIZMA_noDelta). Returns O:[B,H,T,d], S_end:[B,H,d,d]."""
    B, H, T, d = q.shape
    dv = v.shape[-1]                     # value-dim-aware init -> RECTANGULAR state S in R^{d_v x d_k}
    if S0 is None:                       #   (d_k=d). Byte-identical when d_v == d_k (every existing
        S = torch.zeros(B, H, dv, d, dtype=q.dtype, device=q.device)   # call); enables feature-map/GDM
    else:
        S = S0
    gated = alpha is not None
    erase = (write_mode == "delta")
    outs = []
    for c0 in range(0, T, chunk):
        c1 = min(c0 + chunk, T)
        C = c1 - c0
        Kc = k[:, :, c0:c1]                      # [B,H,C,d]
        Vc = v[:, :, c0:c1]
        Qc = q[:, :, c0:c1]
        Bc = beta[:, :, c0:c1]                    # [B,H,C]
        if gated:
            Ac = alpha[:, :, c0:c1]               # [B,H,C]  in [0.5,1]
            # within-chunk cumulative decay in LOG space (avoids float32 underflow of gamma over
            # long chunks: exp(cumsum(log a)) -> 1e-20 at C=64, and dividing two underflowed
            # exponentials destroys the recurrence). All gamma-RATIOS are formed as exp(log-diff),
            # which is <=1 on the strictly-lower region actually used -> stable, no clamp.
            logA = torch.log(Ac.clamp_min(1e-6))
            clog = torch.cumsum(logA, dim=-1)                    # [B,H,C]  log gamma_i (post i)
            clog_prev = clog - logA                              # log gamma_{i-1} (pre i)
            KK = torch.matmul(Kc, Kc.transpose(-1, -2))          # [B,H,C,C]  k_i·k_j
            ratio = torch.exp(clog[..., :, None] - clog[..., None, :])         # gamma_i/gamma_j
            A = torch.tril(Bc[..., :, None] * (KK * ratio), -1) if erase else torch.zeros_like(KK)
            KS0 = torch.matmul(Kc, S.transpose(-1, -2))          # [B,H,C,d]  (k_i^T S0^T)
            gamma = torch.exp(clog)[..., None]                   # [B,H,C,1] absolute (genuine small)
            rhs = Bc[..., None] * (Vc - gamma * KS0) if erase else Bc[..., None] * Vc
            U = _solve_unit_lower(A, rhs)                        # [B,H,C,d]
            # reads are PRE-write -> decayed to gamma_{i-1}
            read_ratio = torch.exp(clog_prev[..., :, None] - clog[..., None, :])   # gamma_{i-1}/gamma_j
            O_inter = torch.exp(clog_prev)[..., None] * torch.matmul(Qc, S.transpose(-1, -2))
            QK = torch.matmul(Qc, Kc.transpose(-1, -2)) * read_ratio
            O_intra = torch.matmul(torch.tril(QK, -1), U)
            Oc = O_inter + O_intra
            # state carry: S_end = gamma_C S0 + sum_i (gamma_C/gamma_i) u_i k_i^T (ratio <=1 -> stable)
            clogC = clog[..., -1:]                               # [B,H,1]
            gC = torch.exp(clogC)[..., None]                     # [B,H,1,1]
            scale = torch.exp(clogC - clog)                      # [B,H,C]  gamma_C/gamma_i <= 1
            S = gC * S + torch.matmul((scale[..., None] * U).transpose(-1, -2), Kc)
        else:
            KK = torch.matmul(Kc, Kc.transpose(-1, -2))          # [B,H,C,C]
            A = torch.tril(Bc[..., :, None] * KK, -1) if erase else torch.zeros_like(KK)
            KS0 = torch.matmul(Kc, S.transpose(-1, -2))          # [B,H,C,d]
            rhs = Bc[..., None] * (Vc - KS0) if erase else Bc[..., None] * Vc
            U = _solve_unit_lower(A, rhs)                        # [B,H,C,d]
            O_inter = torch.matmul(Qc, S.transpose(-1, -2))      # [B,H,C,d]
            QK = torch.matmul(Qc, Kc.transpose(-1, -2))
            O_intra = torch.matmul(torch.tril(QK, -1), U)
            Oc = O_inter + O_intra
            S = S + torch.matmul(U.transpose(-1, -2), Kc)        # S0 + U^T K
        outs.append(Oc)
    return torch.cat(outs, dim=2), S


if __name__ == "__main__":
    torch.manual_seed(0)
    # cover the PRODUCTION regime: chunk=64, long T, the alpha=0.5 decay floor (worst-case
    # underflow), and the additive ablation. (The old test used chunk=16/T=40 and HID a float32
    # underflow bug in the gated chunked path — never weaken these settings.)
    devs = ["cpu", "mps"] if torch.backends.mps.is_available() else ["cpu"]
    cases = [("pure", None, "delta"), ("gated~U", "rand", "delta"),
             ("gated.5floor", "floor", "delta"), ("additive", None, "additive"),
             ("gated-additive", "rand", "additive")]
    allok = True
    for name, amode, wmode in cases:
        for dev in devs:
            B, H, T, d, C = 2, 3, 256, 16, 64
            q = torch.randn(B, H, T, d, device=dev)
            k = torch.randn(B, H, T, d, device=dev); k = k / k.norm(dim=-1, keepdim=True)
            v = torch.randn(B, H, T, d, device=dev)
            beta = torch.rand(B, H, T, device=dev) * 0.99
            if amode == "rand":
                alpha = 0.5 + 0.5 * torch.rand(B, H, T, device=dev)
            elif amode == "floor":
                alpha = torch.full((B, H, T), 0.5, device=dev)     # worst-case sustained decay
            else:
                alpha = None
            Oref, Sref = _delta_reference(q, k, v, beta, alpha, write_mode=wmode)
            Och, Sch = chunked_delta(q, k, v, beta, alpha, chunk=C, write_mode=wmode)
            do = (Oref - Och).abs().max().item(); ds = (Sref - Sch).abs().max().item()
            ok = max(do, ds) < 1e-4; allok &= ok
            print(f"[{name:<14} {dev}] C={C} T={T} max|dO|={do:.2e} max|dS|={ds:.2e}  "
                  f"{'OK' if ok else 'MISMATCH'}")
    # RECTANGULAR state (d_v != d_k): the feature-map / GlobalDeltaMemory contract. The chunked
    # WY/UT form must equal the naive recurrence when keys/queries have a different last-dim than
    # values (state S in R^{d_v x d_k}). This is the load-bearing guarantee for the capacity levers.
    for dev in devs:
        B, H, T, dk, dv, C = 2, 3, 200, 48, 16, 64
        q = torch.randn(B, H, T, dk, device=dev)
        k = torch.randn(B, H, T, dk, device=dev); k = k / k.norm(dim=-1, keepdim=True)
        v = torch.randn(B, H, T, dv, device=dev)
        beta = torch.rand(B, H, T, device=dev) * 0.99
        for amode, wmode in [(None, "delta"), ("rand", "delta"), (None, "additive")]:
            alpha = (0.5 + 0.5 * torch.rand(B, H, T, device=dev)) if amode == "rand" else None
            Oref, Sref = _delta_reference(q, k, v, beta, alpha, write_mode=wmode)
            Och, Sch = chunked_delta(q, k, v, beta, alpha, chunk=C, write_mode=wmode)
            do = (Oref - Och).abs().max().item(); ds = (Sref - Sch).abs().max().item()
            ok = max(do, ds) < 1e-4; allok &= ok
            print(f"[rect dv={dv}!=dk={dk} {dev} {wmode}{'/gated' if amode else ''}] "
                  f"max|dO|={do:.2e} max|dS|={ds:.2e}  {'OK' if ok else 'MISMATCH'}")
    print("ALL OK" if allok else "FAILURES PRESENT")


In [ ]:
%%writefile seq/transformer.py
"""
A clean, properly-implemented decoder-only Transformer baseline (NOT a strawman).
Causal multi-head self-attention via scaled_dot_product_attention, RMSNorm pre-norm,
SwiGLU MLP, learned absolute positions. This is the honest reference Prizma-Seq must match
at equal parameter count.
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TFConfig:
    vocab: int = 64
    d_model: int = 128
    n_layers: int = 2
    n_heads: int = 4
    d_ff: int = None          # default 8/3 * d_model rounded (SwiGLU keeps ~4x FLOPs)
    max_len: int = 1024
    dropout: float = 0.0
    tie_embeddings: bool = True
    rope: bool = True         # use RoPE (parameter-free) instead of a learned pos-embedding

    def __post_init__(self):
        if self.d_ff is None:
            self.d_ff = int(round(8 / 3 * self.d_model / 8) * 8)


def _tf_rope_cache(T, hd, device, dtype):
    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device, dtype=torch.float32) / hd))
    ang = torch.outer(torch.arange(T, device=device, dtype=torch.float32), inv)
    return torch.cos(ang).to(dtype), torch.sin(ang).to(dtype)


def _tf_apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        n = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return n * self.w


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: TFConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.nh = cfg.n_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.drop = cfg.dropout
        self.rope = cfg.rope

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.nh, self.hd).transpose(1, 2)
        k = k.view(B, T, self.nh, self.hd).transpose(1, 2)
        v = v.view(B, T, self.nh, self.hd).transpose(1, 2)
        if self.rope:
            cos, sin = _tf_rope_cache(T, self.hd, x.device, x.dtype)
            q = _tf_apply_rope(q, cos, sin)
            k = _tf_apply_rope(k, cos, sin)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.drop if self.training else 0.0)
        o = o.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(o)

    @torch.no_grad()
    def step(self, x_t, cache):
        """x_t:[B,1,C]; cache=(K,V) each [B,nh,t,hd] or None. KV-cached O(t) decode."""
        B, _, C = x_t.shape
        q, k, v = self.qkv(x_t).split(C, dim=2)
        q = q.view(B, 1, self.nh, self.hd).transpose(1, 2)
        k = k.view(B, 1, self.nh, self.hd).transpose(1, 2)
        v = v.view(B, 1, self.nh, self.hd).transpose(1, 2)
        t = 0 if cache is None else cache[0].shape[2]
        if self.rope:
            cos, sin = _tf_rope_cache(t + 1, self.hd, x_t.device, x_t.dtype)
            q = _tf_apply_rope(q, cos[t:t + 1], sin[t:t + 1])
            k = _tf_apply_rope(k, cos[t:t + 1], sin[t:t + 1])
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=False)   # q attends all cached (<=t)
        o = o.transpose(1, 2).reshape(B, 1, C)
        return self.proj(o), (k, v)


class SwiGLU(nn.Module):
    def __init__(self, cfg: TFConfig):
        super().__init__()
        self.w1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w2 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.wo = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.wo(F.silu(self.w1(x)) * self.w2(x))


class Block(nn.Module):
    def __init__(self, cfg: TFConfig):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.n2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

    @torch.no_grad()
    def step(self, x_t, cache):
        o, cache = self.attn.step(self.n1(x_t), cache)
        x_t = x_t + o
        return x_t + self.mlp(self.n2(x_t)), cache


class Transformer(nn.Module):
    def __init__(self, cfg: TFConfig):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab, cfg.d_model)
        self.pos = None if cfg.rope else nn.Embedding(cfg.max_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.nf = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        if cfg.tie_embeddings:
            self.head.weight = self.tok.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx)
        if self.pos is not None:
            x = x + self.pos(torch.arange(T, device=idx.device))[None]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.nf(x))

    @torch.no_grad()
    def init_state(self, batch, device):
        return [None for _ in self.blocks]              # per-layer KV cache (grows with t)

    @torch.no_grad()
    def step(self, tok, state):
        """tok:[B,1] -> (logits[B,1,V], new_state). KV-cached: O(t) compute, O(t) memory at step t."""
        x = self.tok(tok)
        if self.pos is not None:
            p = 0 if state[0] is None else state[0][0].shape[2]
            x = x + self.pos(torch.tensor([p], device=tok.device))[None]
        new = []
        for blk, c in zip(self.blocks, state):
            x, c2 = blk.step(x, c)
            new.append(c2)
        return self.head(self.nf(x)), new


if __name__ == "__main__":
    from common import param_count
    cfg = TFConfig(vocab=64, d_model=128, n_layers=2, n_heads=4)
    m = Transformer(cfg)
    x = torch.randint(0, 64, (2, 32))
    print("logits", tuple(m(x).shape), "params", param_count(m))


In [ ]:
%%writefile seq/prizma_seq.py
"""
Prizma-Seq — a Predictive-Coding Gated-DeltaNet sequence model (committee spec PRIZMA_SEQ_SPEC.md).

Mixer = a carried associative workspace state S_t in R^{d_h x d_h} per head, updated by a
precision-gated targeted erase-and-write (the delta rule), which is exactly ONE gradient step on
Prizma's per-token free energy F_t(S)=1/2||v_t - S k_t||^2.  Read = S_{t-1} q_t (recognition-by-
reconstruction, strictly pre-write/causal) + a small exact local window head. FFN is byte-identical
to the Transformer baseline so ONLY the mixer differs.

Honest design note: the write gate beta is INPUT-dependent (sigma(W_beta x_t)), which keeps the
chunk-parallel training form valid. Surprise-proportionality is intrinsic: the delta write is
u_t = beta_t * (v_t - S_{t-1} k_t) = beta_t * epsilon_t — it writes the prediction error itself
(Prizma's dW ~ (Pi*eps) (x) r). An optional surprise-gated variant (two-pass) is exposed for B6.
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from .transformer import RMSNorm, SwiGLU, TFConfig
from .delta import chunked_delta


@dataclass
class PrizmaSeqConfig:
    vocab: int = 64
    d_model: int = 64
    n_layers: int = 2
    n_heads: int = 2
    chunk: int = 64
    window: int = 16
    d_ff: int = None
    max_len: int = 1024
    rope: bool = False           # delta keys/queries are POSITION-FREE (DeltaNet/Mamba standard):
                                 #   RoPE makes the recall dot-product distance-dependent and
                                 #   scrambles content-based recall. Position comes from the conv +
                                 #   causal write-order. (RoPE on state keys empirically blocks MQAR.)
    beta_cap: float = 0.99
    gated: bool = False          # data-dependent forget gate alpha (Gated-DeltaNet); off for diagnostics
    learned_pos: bool = False    # add a learned absolute pos-embedding (used for char-LM parity)
    short_conv: int = 4          # short causal depthwise conv before qkv (Mamba/Based/DeltaNet std);
                                 #   lets a token's k/v encode its predecessor -> enables recall. 0=off.
    # --- ablation knobs (B6) ---
    precision_gate: str = "input"   # 'input' (sigma(W_beta x)) | 'uniform' | 'random' (B6 controls)
    write_mode: str = "delta"       # 'delta' (targeted erase-and-write) | 'additive' (linear-attn)
    use_workspace: bool = True      # False -> no carried state (window head only)
    use_window: bool = True         # False -> no local window head (state only)
    route_readout: bool = True      # False -> read state with a FIXED (input-independent) query
                                    #          (B6 noRouteReadout: kills content-based recall)
    # --- capacity lever: parameter-free quadratic key/query feature map (committee R1, rank #1) - #
    feat_map: str = "none"          # 'none' | 'quad2'. 'quad2' expands the DELTA keys/queries from
                                    #   d_h to d_phi = d_h + feat_n2 via FIXED random quadratic
                                    #   monomials -> rectangular carried state S in R^{d_h x d_phi},
                                    #   raising associative-recall key-rank toward D=128 at ZERO
                                    #   trainable params (the indices are buffers, not Parameters)
                                    #   with O(1)/constant-in-n inference intact (state stays fixed).
    feat_n2: int = 96               # number of quadratic monomials (d_phi = d_h + feat_n2)

    def __post_init__(self):
        if self.d_ff is None:
            self.d_ff = int(round(8 / 3 * self.d_model / 8) * 8)
        assert self.d_model % self.n_heads == 0
        self.d_h = self.d_model // self.n_heads
        assert self.d_h % 2 == 0, "d_h must be even for RoPE"
        assert self.feat_map in ("none", "quad2", "rand_linear")
        # d_phi = delta key/query dim after the optional feature map (= d_h when 'none').
        # 'rand_linear' = a FIXED random linear map d_h->d_phi (a CONTROL: it stays in a d_h-rank
        # subspace so it must give NO capacity gain, proving the quad2 MONOMIALS are what help).
        self.d_phi = self.d_h + (self.feat_n2 if self.feat_map in ("quad2", "rand_linear") else 0)


# ----------------------------------- RoPE ------------------------------------------------- #
def _rope_cache(T, d_h, device, dtype, offset=0):
    inv = 1.0 / (10000 ** (torch.arange(0, d_h, 2, device=device, dtype=torch.float32) / d_h))
    pos = torch.arange(offset, offset + T, device=device, dtype=torch.float32)
    ang = torch.outer(pos, inv)                       # [T, d_h/2]
    return torch.cos(ang).to(dtype), torch.sin(ang).to(dtype)


def _apply_rope(x, cos, sin):
    # x: [B,H,T,d_h]; cos/sin: [T,d_h/2]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    rx1 = x1 * cos - x2 * sin
    rx2 = x1 * sin + x2 * cos
    out = torch.empty_like(x)
    out[..., 0::2] = rx1
    out[..., 1::2] = rx2
    return out


def _l2(x, eps=1e-6):
    return x / (x.norm(dim=-1, keepdim=True) + eps)


# --------------------------------- the block ---------------------------------------------- #
class PrizmaSeqBlock(nn.Module):
    def __init__(self, cfg: PrizmaSeqConfig):
        super().__init__()
        self.cfg = cfg
        d, H, dh = cfg.d_model, cfg.n_heads, cfg.d_h
        self.H, self.dh = H, dh
        self.norm1 = RMSNorm(d)
        self.kc = cfg.short_conv
        if self.kc > 0:
            self.conv = nn.Conv1d(d, d, self.kc, groups=d, bias=True)   # depthwise causal short conv
        self.W_qkv = nn.Linear(d, 3 * d, bias=False)
        self.W_beta = nn.Linear(d, H, bias=True)
        self.beta_logit = nn.Parameter(torch.zeros(H))      # uniform (input-independent) gate, B6
        self.q_fixed = nn.Parameter(torch.randn(H, dh) * 0.02)   # noRouteReadout fixed query, B6
        self.W_alpha = nn.Linear(d, H, bias=True) if cfg.gated else None
        self.W_o = nn.Linear(d, d, bias=False)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(TFConfig(d_model=d, d_ff=cfg.d_ff))
        self.win_scale = dh ** -0.5
        self.d_phi = cfg.d_phi
        if cfg.feat_map == "quad2":
            # FIXED random quadratic monomials phi(x) = [x ; x[I]*x[J]] -> d_phi. Seeded => a fixed
            # architectural choice (NOT tuned per task; disclosed). Registered as BUFFERS, so
            # param_count is unchanged (byte-identical param-match to the Transformer preserved).
            g = torch.Generator().manual_seed(1234)
            self.register_buffer("feat_I", torch.randint(0, dh, (cfg.feat_n2,), generator=g))
            self.register_buffer("feat_J", torch.randint(0, dh, (cfg.feat_n2,), generator=g))
        elif cfg.feat_map == "rand_linear":
            # CONTROL (committee): fixed random linear map d_h -> d_phi. rank <= d_h, so it CANNOT
            # raise recall key-rank -> expected NO gain over 'none'. Buffer => param_count unchanged.
            g = torch.Generator().manual_seed(1234)
            self.register_buffer("W_rand", torch.randn(dh, self.d_phi, generator=g) * (dh ** -0.5))

    def _apply_conv(self, x):
        """Causal depthwise short conv + SiLU on the normed input (Mamba/Based-style). x:[B,T,d]."""
        if self.kc == 0:
            return x
        xc = F.pad(x.transpose(1, 2), (self.kc - 1, 0))          # left-pad -> causal
        return F.silu(self.conv(xc).transpose(1, 2))             # [B,T,d]

    def _phi(self, x):
        """Quadratic key/query feature map for the DELTA path only. x:[...,d_h] (already L2-normed)
        -> [...,d_phi]. Identity when feat_map='none'; else _l2([x ; x[I]*x[J]]) over FIXED random
        monomial indices, which escapes the d_h subspace and cuts associative-recall crosstalk
        (D=128: ~0.141 -> ~0.076, ~matching a true d=128 key set). The final _l2 preserves ||k||=1,
        the invariant the delta kernel relies on. The local-window head keeps the linear L2 keys."""
        if self.cfg.feat_map == "none":
            return x
        if self.cfg.feat_map == "rand_linear":
            return _l2(x @ self.W_rand)               # control: rank <= d_h -> no capacity gain
        two = x[..., self.feat_I] * x[..., self.feat_J]
        return _l2(torch.cat([x, two], dim=-1))

    def _encode(self, x, cos, sin):
        B, T, _ = x.shape
        qkv = self.W_qkv(x).view(B, T, self.H, 3, self.dh)
        q, k, v = qkv.unbind(3)                                   # each [B,T,H,dh]
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)   # [B,H,T,dh]
        if self.cfg.rope:
            q = _apply_rope(q, cos, sin)
            k = _apply_rope(k, cos, sin)
        q, k = _l2(q), _l2(k)
        B, T = x.shape[0], x.shape[1]
        if not self.cfg.route_readout:    # B6: read with a FIXED query -> no content-based recall
            q = _l2(self.q_fixed[None, :, None, :].expand(B, self.H, T, self.dh).contiguous())
        if self.cfg.precision_gate == "uniform":
            beta = torch.sigmoid(self.beta_logit)[None, :, None].expand(B, self.H, T) * self.cfg.beta_cap
        elif self.cfg.precision_gate == "random":   # input-independent random write gate (B6 control)
            beta = torch.rand(B, self.H, T, device=x.device, dtype=x.dtype) * self.cfg.beta_cap
        else:
            beta = torch.sigmoid(self.W_beta(x)).transpose(1, 2) * self.cfg.beta_cap   # [B,H,T]
        if self.W_alpha is not None:
            alpha = torch.sigmoid(self.W_alpha(x)).transpose(1, 2)                 # [B,H,T]
            alpha = 0.5 + 0.5 * alpha           # keep decay in [0.5,1] (stability)
        else:
            alpha = None
        return q, k, v, beta, alpha

    def _window(self, q, k, v):
        """Exact causal attention restricted to the last `w` tokens (incl. self). Fused SDPA + band
        mask (fast on MPS); scaling 1/sqrt(d_h) matches the streaming step() path."""
        T = q.shape[2]
        w = self.cfg.window
        idx = torch.arange(T, device=q.device)
        band = (idx[None, :] <= idx[:, None]) & (idx[None, :] > idx[:, None] - w)  # [T,T] True=allow
        mask = torch.zeros(T, T, device=q.device, dtype=q.dtype).masked_fill(~band, float("-inf"))
        return F.scaled_dot_product_attention(q, k, v, attn_mask=mask)            # [B,H,T,dh]

    def forward(self, h):
        B, T, d = h.shape
        x = self._apply_conv(self.norm1(h))
        cos, sin = _rope_cache(T, self.dh, h.device, h.dtype) if self.cfg.rope else (None, None)
        q, k, v, beta, alpha = self._encode(x, cos, sin)
        o = torch.zeros(B, self.H, T, self.dh, device=h.device, dtype=h.dtype)
        if self.cfg.use_workspace:
            # delta state keyed by phi(q),phi(k) (dim d_phi); values stay d_h -> state [B,H,d_h,d_phi]
            o_delta, _ = chunked_delta(self._phi(q), self._phi(k), v, beta, alpha,
                                       chunk=self.cfg.chunk, write_mode=self.cfg.write_mode)  # [B,H,T,d_h]
            o = o + o_delta
        if self.cfg.use_window:
            o = o + self._window(q, k, v)            # window head keeps the LINEAR L2 keys (dim d_h)
        o = o.transpose(1, 2).reshape(B, T, d)                                     # merge heads
        h = h + self.W_o(o)
        h = h + self.mlp(self.norm2(h))
        return h

    # ---- O(1)-per-step inference path (for B5 latency / true streaming) ---- #
    @torch.no_grad()
    def step(self, h_t, state):
        """h_t:[B,1,d]; state=(S, ring_k, ring_v, conv_ring, pos). Returns o_t, new_state. O(1)."""
        B = h_t.shape[0]
        S, rk, rv, cring, pos = state
        xin = self.norm1(h_t)                                    # [B,1,d]
        if self.kc > 0:
            buf = torch.cat([cring, xin], dim=1)                 # [B,kc,d]
            w = self.conv.weight.squeeze(1)                      # [d,kc]
            xc = (buf.transpose(1, 2) * w).sum(-1) + self.conv.bias   # [B,d]
            x = F.silu(xc)[:, None, :]                           # [B,1,d]
            cring = buf[:, 1:, :]                                # keep last kc-1
        else:
            x = xin
        cos, sin = _rope_cache(1, self.dh, h_t.device, h_t.dtype, offset=pos) if self.cfg.rope else (None, None)
        q, k, v, beta, alpha = self._encode(x, cos, sin)         # [B,H,1,dh], beta [B,H,1]
        q1, k1, v1 = q[:, :, 0], k[:, :, 0], v[:, :, 0]          # [B,H,dh] (linear L2; window keys)
        q1p, k1p = self._phi(q)[:, :, 0], self._phi(k)[:, :, 0]  # [B,H,d_phi] (delta state keys)
        b1 = beta[:, :, 0]                                       # [B,H]
        a1 = alpha[:, :, 0] if alpha is not None else torch.ones_like(b1)
        o_delta = torch.einsum("bhij,bhj->bhi", S, q1p)          # pre-write read S_{t-1} phi(q)
        Sk = torch.einsum("bhij,bhj->bhi", S, k1p)               # [B,H,d_h]
        u = b1[..., None] * (v1 - a1[..., None] * Sk)            # [B,H,d_h]
        S = a1[..., None, None] * S + torch.einsum("bhi,bhj->bhij", u, k1p)   # [B,H,d_h,d_phi]
        # window ring
        rk = torch.cat([rk, k1[:, :, None]], dim=2)[:, :, -self.cfg.window:]
        rv = torch.cat([rv, v1[:, :, None]], dim=2)[:, :, -self.cfg.window:]
        sc = torch.einsum("bhd,bhwd->bhw", q1, rk) * self.win_scale
        aw = torch.softmax(sc, dim=-1)
        o_win = torch.einsum("bhw,bhwd->bhd", aw, rv)
        o = (o_delta + o_win).reshape(B, 1, -1)
        h = h_t + self.W_o(o)
        h = h + self.mlp(self.norm2(h))
        return h, (S, rk, rv, cring, pos + 1)


class PrizmaSeqLM(nn.Module):
    def __init__(self, cfg: PrizmaSeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab, cfg.d_model)
        self.pos = nn.Embedding(cfg.max_len, cfg.d_model) if cfg.learned_pos else None
        self.blocks = nn.ModuleList([PrizmaSeqBlock(cfg) for _ in range(cfg.n_layers)])
        self.nf = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.head.weight = self.tok.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        h = self.tok(idx)
        if self.pos is not None:
            h = h + self.pos(torch.arange(T, device=idx.device))[None]
        for blk in self.blocks:
            h = blk(h)
        return self.head(self.nf(h))

    @torch.no_grad()
    def init_state(self, batch, device):
        st = []
        kc1 = max(self.cfg.short_conv - 1, 0)
        for _ in self.blocks:
            S = torch.zeros(batch, self.cfg.n_heads, self.cfg.d_h, self.cfg.d_phi, device=device)
            rk = torch.zeros(batch, self.cfg.n_heads, 0, self.cfg.d_h, device=device)
            rv = torch.zeros(batch, self.cfg.n_heads, 0, self.cfg.d_h, device=device)
            cring = torch.zeros(batch, kc1, self.cfg.d_model, device=device)
            st.append((S, rk, rv, cring, 0))
        return st

    @torch.no_grad()
    def step(self, tok, state):
        """tok:[B,1] -> (logits[B,1,V], new_state). O(1) in sequence length."""
        h = self.tok(tok)
        if self.pos is not None:
            p = state[0][4] if state else 0
            h = h + self.pos(torch.tensor([p], device=tok.device))[None]
        new = []
        for blk, st in zip(self.blocks, state):
            h, st2 = blk.step(h, st)
            new.append(st2)
        return self.head(self.nf(h)), new


def prizma_seq_factory(d_model=64, n_layers=2, n_heads=2, **kw):
    def f(vocab, max_len):
        return PrizmaSeqLM(PrizmaSeqConfig(vocab=vocab, d_model=d_model, n_layers=n_layers,
                                         n_heads=n_heads, max_len=max_len + 8, **kw))
    return f


if __name__ == "__main__":
    from .common import param_count, get_device
    dev = get_device()
    # O(1) GUARD (committee guardrail): for BOTH feat_map settings the streaming step() must equal
    # the parallel forward() to <1e-4, AND param_count must be identical (feature map = 0 params).
    for feat in ("none", "quad2"):
        cfg = PrizmaSeqConfig(vocab=64, d_model=64, n_layers=2, n_heads=2, feat_map=feat)
        m = PrizmaSeqLM(cfg).to(dev)
        x = torch.randint(0, 64, (2, 48), device=dev)
        y = m(x)
        m.train(False)
        st = m.init_state(2, dev)
        outs = []
        for t in range(x.shape[1]):
            lg, st = m.step(x[:, t:t + 1], st)
            outs.append(lg)
        yo = torch.cat(outs, dim=1)
        d = (y - yo).abs().max().item()
        print(f"[feat_map={feat:<5} d_phi={cfg.d_phi}] forward {tuple(y.shape)} params {param_count(m)} "
              f"step-vs-forward max|d|={d:.2e} {'OK' if d < 1e-4 else 'MISMATCH'}")


In [ ]:
%%writefile seq/tasks.py
"""
Canonical attention-diagnostic synthetic tasks — the suite the field uses to decide whether a
non-attention architecture can really stand in for a Transformer.

All tasks share one causal next-token interface:
    sample(batch, device) -> (inputs[B,T] long, targets[B,T] long, mask[B,T] {0,1} float)
Loss/accuracy are computed only on masked positions. Token id 0 is a reserved filler/pad.

  * MQAR          Multi-Query Associative Recall (Arora et al. 2023, Zoology/Based). THE test that
                  separates real attention-alternatives from impostors: in-context key->value
                  lookup for many queries, over a gap. Linear models with fixed state struggle as
                  #pairs grows; attention solves it trivially.
  * AssocRecall   single-query AR (Ba et al.; Hyena) — easier sanity version of MQAR.
  * SelectiveCopy Mamba's selective-copy: copy the data tokens in order, ignoring interspersed
                  fillers. Requires INPUT-DEPENDENT gating (content-selective memory).
  * Induction     in-context induction-head probe: [.. A B .. A] -> predict B. The ICL primitive.
"""
from __future__ import annotations

import torch


def _distinct_per_row(B, n, lo, hi, device):
    """n distinct ints in [lo,hi) for each of B rows -> (B,n) long. hi-lo must be >= n."""
    span = hi - lo
    perm = torch.argsort(torch.rand(B, span, device=device), dim=1)[:, :n]
    return perm + lo


class MQAR:
    """Multi-Query Associative Recall (Zoology/Based standard, dense). vocab in [1,V); 0 filler.
    Sequence = [k0 v0 ... k_{D-1} v_{D-1}]  (+optional filler gap)  [q_1 q_2 ... q_M], where each
    query q is one of the bound keys. The TARGET at each query position is that key's bound value
    (NOT a next token — the answer never appears in the input), masked to query positions only.
    Dense queries give rich supervision; capacity is set by the #bindings D (the B1b axis)."""

    def __init__(self, vocab=64, num_pairs=8, num_queries=None, gap=0, key_frac=0.5):
        # DISJOINT token ranges: keys in [1, n_key), values in [n_key, vocab). This removes
        # key/value collisions so recall is unambiguous and the capacity ceiling (B1b) is clean.
        self.n_key = max(num_pairs + 1, int(vocab * key_frac))
        assert self.n_key - 1 >= num_pairs, "need >= num_pairs distinct keys in the key range"
        assert vocab - self.n_key >= 2, "need a value range"
        self.vocab = vocab
        self.D = num_pairs
        self.M = num_queries if num_queries is not None else max(2 * num_pairs, 8)
        self.gap = gap
        self.seq_len = 2 * num_pairs + gap + self.M
        self.name = f"MQAR(V={vocab},keys<{self.n_key},pairs={num_pairs},q={self.M},gap={gap})"

    def sample(self, B, device):
        D, M, V = self.D, self.M, self.vocab
        keys = _distinct_per_row(B, D, 1, self.n_key, device)     # (B,D) distinct keys
        vals = torch.randint(self.n_key, V, (B, D), device=device)  # (B,D) values (disjoint range)
        ctx = torch.empty(B, 2 * D, dtype=torch.long, device=device)
        ctx[:, 0::2] = keys
        ctx[:, 1::2] = vals
        qi = torch.randint(0, D, (B, M), device=device)           # which binding each query hits
        qk = torch.gather(keys, 1, qi)                            # query keys (the input)
        qa = torch.gather(vals, 1, qi)                            # bound values (the target)
        if self.gap > 0:
            filler = torch.zeros(B, self.gap, dtype=torch.long, device=device)
            inp = torch.cat([ctx, filler, qk], dim=1)
        else:
            inp = torch.cat([ctx, qk], dim=1)
        T = inp.shape[1]
        tgt = torch.zeros(B, T, dtype=torch.long, device=device)
        mask = torch.zeros(B, T, device=device)
        qstart = 2 * D + self.gap
        tgt[:, qstart:] = qa                                      # recall target at each query pos
        mask[:, qstart:] = 1.0
        return inp, tgt, mask


class AssocRecall:
    """Single-query associative recall: many KV pairs, one query at the end."""

    def __init__(self, vocab=64, num_pairs=16, gap=0):
        self.inner = MQAR(vocab=vocab, num_pairs=num_pairs, num_queries=1, gap=gap)
        self.vocab = vocab
        self.seq_len = self.inner.seq_len
        self.name = f"AssocRecall(V={vocab},pairs={num_pairs},gap={gap})"

    def sample(self, B, device):
        return self.inner.sample(B, device)


class SelectiveCopy:
    """Copy the K data tokens (in order) out of an L-slot memory region full of fillers.
    vocab: filler=0, data in [1, vocab-1), marker=vocab-1.
    input = [memory(L) with K data tokens scattered, marker, d1, d2, ..., dK]
    predict d1..dK at positions marker..d_{K-1} (next-token), masked there."""

    def __init__(self, vocab=32, mem_len=64, n_data=16, fixed=False):
        assert n_data <= mem_len
        self.vocab = vocab
        self.L = mem_len
        self.K = n_data
        self.fixed = fixed     # True -> data at fixed evenly-spaced positions (control variant)
        self.marker = vocab - 1
        self.seq_len = mem_len + 1 + n_data
        self.name = f"SelectiveCopy(V={vocab},mem={mem_len},k={n_data},{'fixed' if fixed else 'selective'})"

    def sample(self, B, device):
        L, K, V = self.L, self.K, self.vocab
        mem = torch.zeros(B, L, dtype=torch.long, device=device)            # fillers
        if self.fixed:
            base = torch.linspace(0, L - 1, K, device=device).long()        # fixed spacing (control)
            pos = base[None].expand(B, K).clone()
        else:
            pos = torch.argsort(torch.rand(B, L, device=device), dim=1)[:, :K]  # K random slots/row
        pos, _ = torch.sort(pos, dim=1)                                     # left-to-right order
        data = torch.randint(1, V - 1, (B, K), device=device)              # data tokens
        mem.scatter_(1, pos, data)
        marker = torch.full((B, 1), self.marker, dtype=torch.long, device=device)
        inp = torch.cat([mem, marker, data], dim=1)                        # teacher-forced output
        T = inp.shape[1]
        tgt = torch.zeros_like(inp)
        tgt[:, :-1] = inp[:, 1:]
        mask = torch.zeros(B, T, device=device)
        mask[:, L:L + K] = 1.0     # positions: marker (predict d1) .. d_{K-1} (predict dK)
        return inp, tgt, mask


class Induction:
    """In-context induction: prefix contains a unique bigram [A,B]; the final token is A; predict B.
    vocab tokens in [1, vocab); 0 filler. mask=1 only at the final position."""

    def __init__(self, vocab=32, seq_len=64):
        self.vocab = vocab
        self.seq_len = seq_len + 1   # +1 for the trailing query A
        self.name = f"Induction(V={vocab},len={seq_len})"
        self._L = seq_len

    def sample(self, B, device):
        L, V = self._L, self.vocab
        seq = torch.randint(1, V, (B, L), device=device)
        A = torch.randint(1, V, (B,), device=device)
        B_tok = torch.randint(1, V, (B,), device=device)
        # ensure B != A so the answer is informative
        clash = B_tok == A
        B_tok[clash] = (B_tok[clash] % (V - 1)) + 1
        B_tok[B_tok == A] = (A[B_tok == A] % (V - 1)) + 1
        # remove any existing A from the prefix (so A is unique once we place the bigram)
        seq[seq == A[:, None]] = 0
        # also keep the bigram-following slot clean: place [A,B] at a random early position
        ppos = torch.randint(0, L - 2, (B,), device=device)
        ar = torch.arange(B, device=device)
        seq[ar, ppos] = A
        seq[ar, ppos + 1] = B_tok
        # any filler 0 left from removal -> replace with a token guaranteed != A
        filler = (A % (V - 1)) + 1
        zero = seq == 0
        seq = torch.where(zero, filler[:, None].expand_as(seq), seq)
        # but that replacement might have re-introduced A if filler==A (filler!=A by construction)
        # re-place bigram in case a filler overwrote it (positions fixed, so re-assert)
        seq[ar, ppos] = A
        seq[ar, ppos + 1] = B_tok
        query = A[:, None]
        inp = torch.cat([seq, query], dim=1)            # (B, L+1)
        T = inp.shape[1]
        tgt = torch.zeros(B, T, dtype=torch.long, device=device)
        tgt[:, -1] = B_tok
        mask = torch.zeros(B, T, device=device)
        mask[:, -1] = 1.0
        return inp, tgt, mask


class MixedMQAR:
    """MQAR trained over a SPECTRUM of difficulties: each training batch samples the number of KV
    pairs d ~ U[min_pairs, max_pairs] (the standard Zoology training distribution). The easy
    instances supply the gradient that BOOTSTRAPS the recall circuit, so high-D recall becomes
    learnable in feasible compute — where FIXED-high-D training stalls at chance (no foothold for the
    sharp phase transition). EVALUATION is fixed at the target (max_pairs), so the reported number is
    target-D recall. Identical for both models -> fair. n_key is stable across d (= vocab*key_frac
    while d < that), so key/value ranges don't shift with difficulty."""

    def __init__(self, vocab=256, max_pairs=64, num_queries=128, gap=0, min_pairs=1):
        self.vocab = vocab
        self.max_pairs = max_pairs
        self.M = num_queries
        self.gap = gap
        self.min_pairs = min_pairs
        self.seq_len = 2 * max_pairs + gap + num_queries          # max layout (for max_len sizing)
        self.name = f"MixedMQAR(V={vocab},pairs={min_pairs}-{max_pairs},q={num_queries},gap={gap})"

    def _mqar(self, d):
        return MQAR(vocab=self.vocab, num_pairs=d, num_queries=self.M, gap=self.gap)

    def sample(self, B, device):            # TRAINING: a random difficulty per batch
        d = int(torch.randint(self.min_pairs, self.max_pairs + 1, (1,), device=device).item())
        return self._mqar(d).sample(B, device)

    def eval_sample(self, B, device):       # EVAL: fixed at the TARGET difficulty (max_pairs)
        return self._mqar(self.max_pairs).sample(B, device)


TASK_REGISTRY = {
    "mqar": MQAR,
    "mixed_mqar": MixedMQAR,
    "assoc": AssocRecall,
    "selcopy": SelectiveCopy,
    "induction": Induction,
}


if __name__ == "__main__":
    dev = torch.device("cpu")
    for name, cls in TASK_REGISTRY.items():
        t = cls()
        x, y, m = t.sample(4, dev)
        print(f"{t.name:<40} x{tuple(x.shape)} masked={int(m.sum().item())}")
        # sanity: an oracle that knows the mapping would score 1.0; random ~1/vocab


In [ ]:
%%writefile gpu_bench.py
"""Prizma-Seq vs Transformer — the rigorous, scaled D=128 benchmark, for a CUDA GPU (Colab).

Answers, with multi-seed + fair protocol, the questions the committee R2 verdict demands:
  PHASE 1  Find the SMALLEST Transformer scale that genuinely SOLVES MQAR D=128 (the fair arena);
           this is also the committee rank-1 flip-test (does attention solve D=128 with enough
           scale/budget?). Multi-config x recipe x seed -> solve-rate.
  PHASE 2  At that scale S*: matched + FLOP-comparable head-to-head at D=128 (TF vs Prizma-none vs
           Prizma-quad2), >=5 seeds -> solve-rate + median + 95% CI. The headline fair comparison.
  PHASE 3  D-frontier {16,32,64,128,256} at a fixed scale: TF vs Prizma-quad2 vs none (capacity curve).
  PHASE 4  Ablations at D=128: quad2 vs none vs rand_linear control; window on/off (causal attribution).
  PHASE 5  FLOP ledger + MEASURED O(1) decode latency & memory vs sequence length.

All training uses MixedMQAR (mixed-difficulty -> high-D is learnable) + gen-warm + per-model plateau.
Results stream incrementally to $PRIZMA_RESULTS/gpu_bench.json (resumable: completed cells are skipped),
so a Colab disconnect never loses progress. Designed to finish in a few hours on an A100/L4.

Env: set PRIZMA_RESULTS to a Drive-mounted dir for persistence (default ./results).
Run: python3 gpu_bench.py            # all phases
     python3 gpu_bench.py 1 2        # only listed phases
"""
from __future__ import annotations

import json
import math
import os
import sys
import time

import numpy as np
import torch

from seq.common import TrainConfig, train_model, param_count, get_device
from seq.tasks import MixedMQAR, MQAR
from seq.transformer import Transformer, TFConfig
from seq.prizma_seq import PrizmaSeqLM, PrizmaSeqConfig

DEV = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
RES = os.environ.get("PRIZMA_RESULTS", os.path.join(os.path.dirname(__file__), "results"))
os.makedirs(RES, exist_ok=True)
OUT = os.path.join(RES, "gpu_bench.json")
GENWARM = dict(lr=1e-3, warmup=2000, warmup_frac=0.0, min_lr_frac=0.1)


def _load():
    return json.load(open(OUT)) if os.path.exists(OUT) else {}


def _save(d):
    tmp = OUT + ".tmp"
    json.dump(d, open(tmp, "w"), indent=2)
    os.replace(tmp, OUT)


def ci95(xs):
    xs = np.asarray(xs, float)
    if len(xs) < 2:
        return float(xs.mean()), 0.0
    return float(xs.mean()), float(1.96 * xs.std(ddof=1) / math.sqrt(len(xs)))


def _median(xs):
    s = sorted(xs); n = len(s)
    return s[n // 2] if n % 2 else 0.5 * (s[n // 2 - 1] + s[n // 2])


def tf_factory(d, L, H):
    return lambda V, T: Transformer(TFConfig(vocab=V, d_model=d, n_layers=L, n_heads=H, max_len=T + 8, rope=True))


def ps_factory(d, L, H, **kw):
    return lambda V, T: PrizmaSeqLM(PrizmaSeqConfig(vocab=V, d_model=d, n_layers=L, n_heads=H, max_len=T + 8, **kw))


def run_cell(res, cellkey, model_fac, task_fac, cap, seed, recipe=GENWARM, eval_every=2000):
    """Train one (model x task x seed) cell; cache by cellkey; return the record."""
    if cellkey in res and "best" in res[cellkey]:
        return res[cellkey]
    task = task_fac()
    model = model_fac(task.vocab, task.seq_len)
    p = param_count(model)
    cfg = TrainConfig(steps=cap, batch_size=64, log=False, eval_every=eval_every, **recipe)
    t0 = time.time()
    r = train_model(model, task, cfg, DEV, seed=seed)
    rec = {"best": r.best_acc, "plateau": r.steps_to_plateau, "params": p,
           "sec": round(time.time() - t0, 1), "seed": seed, "cap": cap}
    res[cellkey] = rec
    _save(res)
    print(f"   [{cellkey}] best={rec['best']:.3f} plateau@{rec['plateau']} ({rec['sec']}s, {p}p)", flush=True)
    return rec


def solve_stats(recs):
    bests = [r["best"] for r in recs]
    return {"solve_rate": f"{sum(b > 0.9 for b in bests)}/{len(bests)}", "median": round(_median(bests), 4),
            "mean_ci95": [round(x, 4) for x in ci95(bests)], "bests": [round(b, 3) for b in bests]}


# ----------------------------------------------------------------------------------------------- #
def phase1(res):
    """Smallest TF scale that SOLVES MQAR D=128 (fair arena + committee flip-test). Multi-seed."""
    print("\n==== PHASE 1: TF D=128 solving-scale search (mixed-D, gen-warm) ====", flush=True)
    V = 512
    task_fac = lambda: MixedMQAR(vocab=V, max_pairs=128, num_queries=128, gap=0, min_pairs=1)
    configs = {"d64L2H2": (64, 2, 2), "d128L2H4": (128, 2, 4), "d128L4H4": (128, 4, 4), "d256L4H8": (256, 4, 8)}
    seeds = (0, 1, 2)
    summary = {}
    for cname, (d, L, H) in configs.items():
        recs = [run_cell(res, f"p1.TF.{cname}.s{s}", tf_factory(d, L, H), task_fac, 80000, s) for s in seeds]
        summary[cname] = solve_stats(recs)
        print(f"  -> TF {cname}: {summary[cname]}", flush=True)
    res["p1_summary"] = summary; _save(res)
    return summary


def phase2(res, scale=(128, 4, 4), feat_n2=224, seeds=(0, 1, 2, 3, 4)):
    """Head-to-head @ D=128 at scale S*: TF vs Prizma-none vs Prizma-quad2 (>=5 seeds, CI)."""
    d, L, H = scale
    print(f"\n==== PHASE 2: head-to-head @ D=128, scale d{d}L{L}H{H} ({len(seeds)} seeds) ====", flush=True)
    V = 512
    task_fac = lambda: MixedMQAR(vocab=V, max_pairs=128, num_queries=128, gap=0, min_pairs=1)
    arms = {"TF": tf_factory(d, L, H), "Prizma-none": ps_factory(d, L, H),
            "Prizma-quad2": ps_factory(d, L, H, feat_map="quad2", feat_n2=feat_n2)}
    summary = {}
    for aname, fac in arms.items():
        recs = [run_cell(res, f"p2.{aname}.s{s}", fac, task_fac, 80000, s) for s in seeds]
        summary[aname] = solve_stats(recs)
        print(f"  -> {aname}: {summary[aname]}", flush=True)
    res["p2_summary"] = summary; _save(res)
    return summary


def phase3(res, scale=(128, 4, 4), feat_n2=224, seeds=(0, 1, 2)):
    """D-frontier {16,32,64,128,256}: TF vs Prizma-quad2 vs none at a fixed scale."""
    d, L, H = scale
    print(f"\n==== PHASE 3: D-frontier @ scale d{d}L{L}H{H} ====", flush=True)
    arms = {"TF": tf_factory(d, L, H), "Prizma-none": ps_factory(d, L, H),
            "Prizma-quad2": ps_factory(d, L, H, feat_map="quad2", feat_n2=feat_n2)}
    summary = {}
    for D in (16, 32, 64, 128, 256):
        V = max(64, 4 * D)
        task_fac = lambda V=V, D=D: MixedMQAR(vocab=V, max_pairs=D, num_queries=128, gap=0, min_pairs=1)
        cap = 60000 if D <= 64 else 90000
        for aname, fac in arms.items():
            recs = [run_cell(res, f"p3.D{D}.{aname}.s{s}", fac, task_fac, cap, s) for s in seeds]
            summary[f"D{D}.{aname}"] = solve_stats(recs)
            print(f"  -> D={D} {aname}: {summary[f'D{D}.{aname}']}", flush=True)
    res["p3_summary"] = summary; _save(res)
    return summary


def phase4(res, scale=(128, 4, 4), feat_n2=224, seeds=(0, 1, 2)):
    """Ablations @ D=128: quad2 vs none vs rand_linear control; window on/off (causal attribution)."""
    d, L, H = scale
    print(f"\n==== PHASE 4: ablations @ D=128, scale d{d}L{L}H{H} ====", flush=True)
    V = 512
    task_fac = lambda: MixedMQAR(vocab=V, max_pairs=128, num_queries=128, gap=0, min_pairs=1)
    arms = {
        "quad2":            ps_factory(d, L, H, feat_map="quad2", feat_n2=feat_n2),
        "none":             ps_factory(d, L, H),
        "rand_linear":      ps_factory(d, L, H, feat_map="rand_linear", feat_n2=feat_n2),  # control: expect ~none
        "quad2_noWindow":   ps_factory(d, L, H, feat_map="quad2", feat_n2=feat_n2, use_window=False),
    }
    summary = {}
    for aname, fac in arms.items():
        recs = [run_cell(res, f"p4.{aname}.s{s}", fac, task_fac, 80000, s) for s in seeds]
        summary[aname] = solve_stats(recs)
        print(f"  -> {aname}: {summary[aname]}", flush=True)
    res["p4_summary"] = summary; _save(res)
    return summary


def phase5(res, scale=(128, 4, 4), feat_n2=224):
    """Measured decode latency + state memory vs sequence length (the O(1) structural advantage)."""
    d, L, H = scale
    print(f"\n==== PHASE 5: measured O(1) decode latency + memory vs T, scale d{d}L{L}H{H} ====", flush=True)
    V = 512
    tf = Transformer(TFConfig(vocab=V, d_model=d, n_layers=L, n_heads=H, max_len=4200, rope=True)).to(DEV)
    ps = PrizmaSeqLM(PrizmaSeqConfig(vocab=V, d_model=d, n_layers=L, n_heads=H, max_len=4200,
                                   feat_map="quad2", feat_n2=feat_n2)).to(DEV)
    tf.train(False); ps.train(False)
    ns = [128, 256, 512, 1024, 2048, 4096]

    @torch.no_grad()
    def decode_latency(model, n, reps=3, warmup=5):
        lat = []
        for r in range(reps + warmup):
            st = model.init_state(1, DEV); tok = torch.randint(0, V, (1, 1), device=DEV)
            if DEV.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n):
                lg, st = model.step(tok, st); tok = lg[:, -1:].argmax(-1)
            if DEV.type == "cuda": torch.cuda.synchronize()
            if r >= warmup: lat.append(time.time() - t0)
        return float(np.median(lat))

    out = {"seq_lens": ns, "tf_decode_s": {}, "prizma_decode_s": {}}
    for n in ns:
        out["tf_decode_s"][n] = round(decode_latency(tf, n), 4)
        out["prizma_decode_s"][n] = round(decode_latency(ps, n), 4)
        print(f"  n={n:<5} TF(KV)={out['tf_decode_s'][n]:.4f}s  Prizma(O(1))={out['prizma_decode_s'][n]:.4f}s", flush=True)
    # state size (floats): TF KV-cache grows O(n); Prizma state constant
    dh = d // H
    out["tf_kv_floats"] = {n: 2 * L * H * dh * n for n in ns}
    out["prizma_state_floats"] = {n: L * H * dh * (dh + feat_n2) + 2 * L * H * 16 * dh for n in ns}  # state + window ring
    res["p5_latency"] = out; _save(res)
    return out


PHASES = {1: phase1, 2: phase2, 3: phase3, 4: phase4, 5: phase5}


def main():
    wanted = [int(a) for a in sys.argv[1:]] or [1, 2, 3, 4, 5]
    print(f"device={DEV} torch={torch.__version__} results={OUT} phases={wanted}", flush=True)
    if DEV.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
    res = _load()
    for ph in wanted:
        PHASES[ph](res)
    print("\n==== DONE. Summary keys:", [k for k in res if k.endswith("_summary") or k == "p5_latency"], flush=True)
    print(f"saved -> {OUT}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile flop_ledger.py
"""SKEPTIC-C FLOP / throughput ledger. Analytical per-token forward FLOPs for the
matched-param TF vs Prizma-quad2-256 at seq T=384 (the D=128 sweep config), plus a
measured wall-clock throughput probe and a window-head ablation FLOP delta.

Counts MACs*2 = FLOPs. Matmul [m,k]x[k,n] = 2*m*k*n FLOPs. We count per SEQUENCE
(all T tokens) for the training forward, then divide by T for per-token. Attention
is counted at its true O(T^2) training cost (full causal). Prizma's chunked_delta is
counted at its true training cost (the chunk matmuls), and the window head at its
exact banded cost. Decode (step) FLOPs counted separately (the O(1) claim).
"""
import torch
from seq.transformer import Transformer, TFConfig
from seq.prizma_seq import PrizmaSeqLM, PrizmaSeqConfig
from seq.common import param_count

V = 512
T = 384            # MixedMQAR(max_pairs=128): seq_len = 2*128+128 = 384
d = 64
H = 2
dh = d // H        # 32
dff = 168
L = 2

def mm(m, k, n):   # FLOPs of [m,k]@[k,n]
    return 2 * m * k * n

# ----------------------------------------------------------------------------- #
# TRANSFORMER per-sequence forward FLOPs (one layer), then x L.
# ----------------------------------------------------------------------------- #
def tf_layer_flops(T):
    f = {}
    f["qkv_proj"] = mm(T, d, 3 * d)
    # SDPA causal: scores QK^T over causal pairs ~ T^2/2 each, times head dim; full T^2 upper bound
    # count full T*T (SDPA computes full then masks) to be GENEROUS to the TF (over-counts it):
    f["attn_scores"] = H * mm(T, dh, T)        # Q@K^T : [T,dh]x[dh,T] per head
    f["attn_AV"]     = H * mm(T, T, dh)        # A@V
    f["attn_out"]    = mm(T, d, d)             # proj
    f["mlp"]         = mm(T, d, dff) + mm(T, d, dff) + mm(T, dff, d)  # w1,w2,wo
    return f

def tf_layer_flops_causal(T):
    # honest causal count: only lower-triangular pairs ~ T*(T+1)/2
    pairs = T * (T + 1) / 2
    f = {}
    f["qkv_proj"] = mm(T, d, 3 * d)
    f["attn_scores"] = H * 2 * pairs * dh      # sum over causal (i,j) of 2*dh
    f["attn_AV"]     = H * 2 * pairs * dh
    f["attn_out"]    = mm(T, d, d)
    f["mlp"]         = mm(T, d, dff) + mm(T, d, dff) + mm(T, dff, d)
    return f

# ----------------------------------------------------------------------------- #
# Prizma-quad2-256 per-sequence forward FLOPs (one layer).
# d_phi = 256. State S in R^{d_h x d_phi} = 32x256.
# chunked_delta with chunk C=64, nchunks = T/C.
# ----------------------------------------------------------------------------- #
def prizma_layer_flops(T, d_phi, C=64, w=16, kc=4):
    f = {}
    # depthwise causal conv k=4 over d channels: ~ T*d*kc MAC*2
    f["conv"] = 2 * T * d * kc
    # qkv projection (same as TF)
    f["qkv_proj"] = mm(T, d, 3 * d)
    # beta head W_beta: [T,d]x[d,H]
    f["beta"] = mm(T, d, H)
    # phi feature map: quadratic monomials, elementwise — count the d_phi-dh products + l2 norm
    n2 = d_phi - dh
    f["phi_qk"] = H * T * (n2 * 1 + d_phi * 3) * 2  # ~ products + norm; small, generous estimate

    # --- chunked_delta DELTA path, per head, over nchunks chunks of size C ---
    nC = T // C
    # within-chunk matmuls (delta gated=False branch in delta.py):
    #  KK   = Kc @ Kc^T          : [C,d_phi]x[d_phi,C]  -> 2*C*d_phi*C
    #  KS0  = Kc @ S^T           : [C,d_phi]x[d_phi,d_h]-> 2*C*d_phi*d_h
    #  solve_unit_lower (I+A)X=R : ~ C^2*d_h (triangular solve, RHS width d_h) -> ~2*C*C*d_h
    #  O_inter = Qc @ S^T        : [C,d_phi]x[d_phi,d_h]-> 2*C*d_phi*d_h
    #  QK   = Qc @ Kc^T          : [C,d_phi]x[d_phi,C]  -> 2*C*d_phi*C
    #  O_intra = tril(QK) @ U    : [C,C]x[C,d_h]        -> 2*C*C*d_h
    #  S += U^T @ Kc             : [d_h,C]x[C,d_phi]    -> 2*d_h*C*d_phi
    per_chunk = (mm(C, d_phi, C)          # KK
                 + mm(C, d_phi, dh)        # KS0
                 + 2 * C * C * dh          # solve (triangular, approx)
                 + mm(C, d_phi, dh)        # O_inter
                 + mm(C, d_phi, C)         # QK
                 + 2 * C * C * dh          # O_intra
                 + mm(dh, C, d_phi))       # S update
    f["delta_state"] = H * nC * per_chunk

    # --- window head: exact banded attention, window w (incl self). SDPA over [T,T] band ---
    # Prizma forward() builds a FULL [T,T] masked SDPA (see _window) -> pays full T^2 like TF!
    # Count BOTH: (a) the as-implemented full-T^2 forward, (b) the band-only ideal.
    f["window_full_TT"] = H * (mm(T, dh, T) + mm(T, T, dh))   # as-coded forward (full SDPA + mask)
    band_pairs = T * w - w * (w - 1) / 2                       # causal band size
    f["window_band_ideal"] = H * 2 * band_pairs * dh * 2      # scores+AV over band only
    f["out_proj"] = mm(T, d, d)
    f["mlp"] = mm(T, d, dff) + mm(T, d, dff) + mm(T, dff, d)
    return f

def total(fd, exclude=()):  # sum a flop dict excluding some keys
    return sum(v for k, v in fd.items() if k not in exclude)

# embedding/head are shared & identical; ignore for the per-layer mixer comparison,
# but include logits head once for completeness.
head_flops = mm(T, d, V)

print(f"=== Per-SEQUENCE forward FLOPs at T={T}, d={d}, H={H}, d_h={dh}, L={L} ===\n")

# Transformer
tfl = tf_layer_flops_causal(T)
tf_total = L * total(tfl) + head_flops
print("TRANSFORMER per-layer (causal-honest):")
for k, v in tfl.items(): print(f"   {k:<14} {v/1e6:8.2f} MFLOP")
print(f"   LAYER TOTAL    {total(tfl)/1e6:8.2f} MFLOP   x{L} layers + head = {tf_total/1e6:8.2f} MFLOP/seq")
print(f"   per-token: {tf_total/T/1e3:8.1f} kFLOP/token\n")

# Prizma quad2-256
pf = prizma_layer_flops(T, d_phi=256)
# as-coded forward uses window_full_TT; band_ideal is the "if optimally implemented" number
ps_ascoded = total(pf, exclude=("window_band_ideal",))
ps_ideal   = total(pf, exclude=("window_full_TT",))
ps_total_ascoded = L * ps_ascoded + head_flops
ps_total_ideal   = L * ps_ideal + head_flops
print("Prizma-quad2-256 per-layer:")
for k, v in pf.items(): print(f"   {k:<18} {v/1e6:8.2f} MFLOP")
print(f"   LAYER TOTAL as-coded (window_full)  {ps_ascoded/1e6:8.2f} MFLOP")
print(f"   LAYER TOTAL ideal   (window_band)   {ps_ideal/1e6:8.2f} MFLOP")
print(f"   x{L} + head = as-coded {ps_total_ascoded/1e6:8.2f} | ideal {ps_total_ideal/1e6:8.2f} MFLOP/seq")
print(f"   per-token: as-coded {ps_total_ascoded/T/1e3:8.1f} | ideal {ps_total_ideal/T/1e3:8.1f} kFLOP/token\n")

print("=== RATIOS (Prizma / TF), forward FLOPs ===")
print(f"   as-coded forward : {ps_total_ascoded/tf_total:5.2f}x")
print(f"   ideal  forward   : {ps_total_ideal/tf_total:5.2f}x")
print()

# isolate the lever's marginal cost: delta_state for d_phi=256 vs d_phi=32 (none)
pf_none = prizma_layer_flops(T, d_phi=32)
print("=== delta_state cost: d_phi=256 vs d_phi=32 (the quad2 lever's FLOP cost) ===")
print(f"   d_phi=256 delta_state: {pf['delta_state']/1e6:7.2f} MFLOP/layer")
print(f"   d_phi=32  delta_state: {pf_none['delta_state']/1e6:7.2f} MFLOP/layer")
print(f"   lever multiplies delta-state FLOPs by ~{pf['delta_state']/pf_none['delta_state']:.1f}x\n")

# window head share of total
print("=== window head FLOP share (ablation target) ===")
print(f"   window_full   {pf['window_full_TT']/1e6:6.2f} MFLOP/layer = {100*L*pf['window_full_TT']/ps_total_ascoded:4.1f}% of as-coded total")
print(f"   window_band   {pf['window_band_ideal']/1e6:6.2f} MFLOP/layer = {100*L*pf['window_band_ideal']/ps_total_ideal:4.1f}% of ideal total")


## Self-tests (kernels) — should print ALL OK + step==forward <1e-6


In [ ]:
!python -m seq.delta | tail -4
!python -m seq.prizma_seq


## FLOP ledger (analytical disclosure: Prizma/TF forward-FLOP ratio)


In [ ]:
!python flop_ledger.py | grep -E 'per-token|RATIOS|as-coded|ideal'


## Run the full benchmark (phases 1-5). Streams to Drive; hours on A100/L4.


In [ ]:
import sys
sys.argv = ['gpu_bench']      # no args -> all phases (or e.g. ['gpu_bench','1','2'])
import gpu_bench
gpu_bench.main()


## Final summary (also copied here for easy read-back)


In [ ]:
import json, os
d = json.load(open(os.path.join(os.environ['PRIZMA_RESULTS'], 'gpu_bench.json')))
print('===RESULTS_JSON_BEGIN===')
print(json.dumps({k: v for k, v in d.items() if k.endswith('_summary') or k == 'p5_latency'}, indent=2))
print('===RESULTS_JSON_END===')
